In [0]:
import joblib
import pandas as pd
from pyspark.sql import SparkSession

In [0]:
%pip install xgboost

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
spark = SparkSession.builder.getOrCreate()

model_path = "/Volumes/workspace/default/models/high_value_order_xgb_pipeline_final.pkl"
label_encoder_path = "/Volumes/workspace/default/models/high_value_order_label_encoder_final.pkl"

model = joblib.load(model_path)
label_encoder = joblib.load(label_encoder_path)

df = spark.read.table("workspace.default.ecommerce_orders_dataset").sample(fraction=0.18, seed=None)
pdf = df.toPandas()

feature_cols = [
    'Year', 'Month', 'Day', 'Day_Of_Week', 'Quarter',
    'Customer_Age', 'Customer_Gender', 'Country', 'City', 'Customer_Segment',
    'Product_Category', 'Product_Subcategory', 'Brand', 'Discount_Percent',
    'Coupon_Used', 'Payment_Method', 'Device_Type', 'Traffic_Source',
    'Membership_Status', 'Shipping_Method', 'Warehouse_Region',
    'Delivery_Days', 'Review_Rating', 'Season', 'Holiday_Season'
]

X = pdf[feature_cols]

preds = model.predict(X)
decoded_preds = label_encoder.inverse_transform(preds)

pdf["prediction"] = decoded_preds

preds_df = spark.createDataFrame(pdf)
preds_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.ecommerce_predictions")

print("Done. Sample predictions:")
print(pdf[["prediction"]].value_counts())

/databricks/python/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.9.0 when using version 1.6.1. This 

Done. Sample predictions:
prediction
No            4291
Yes           1216
Name: count, dtype: int64


In [0]:
result = spark.read.table("workspace.default.ecommerce_predictions")
result.select("prediction").groupBy("prediction").count().show()

+----------+-----+
|prediction|count|
+----------+-----+
|       Yes| 1216|
|        No| 4291|
+----------+-----+



In [0]:
from pyspark.sql.functions import col

result = spark.read.table("workspace.default.ecommerce_predictions")

# Compare prediction vs actual
result.groupBy("High_Value_Order", "prediction").count().show()

+----------------+----------+-----+
|High_Value_Order|prediction|count|
+----------------+----------+-----+
|             Yes|       Yes| 1111|
|              No|        No| 4022|
|             Yes|        No|  269|
|              No|       Yes|  105|
+----------------+----------+-----+



In [0]:
correct = result.filter(col("High_Value_Order") == col("prediction")).count()
total = result.count()
print("Accuracy on this batch:", correct / total)

Accuracy on this batch: 0.9320864354457963


In [0]:
result = spark.read.table("workspace.default.ecommerce_predictions")

display(result.groupBy("prediction").count())

prediction,count
Yes,1216
No,4291


In [0]:
display(result.select("Order_Date", "Country", "Product_Category", "prediction").limit(100))

Order_Date,Country,Product_Category,prediction
2023-01-01,Germany,Books,No
2023-01-01,United States,Fashion,No
2023-01-01,India,Beauty,No
2023-01-01,Pakistan,Beauty,No
2023-01-01,Australia,Sports,No
2023-01-01,United Kingdom,Electronics,Yes
2023-01-02,United States,Books,No
2023-01-02,Germany,Toys,No
2023-01-02,Germany,Home & Kitchen,No
2023-01-02,Australia,Groceries,No


In [0]:
from pyspark.sql.functions import col

wrong_preds = result.filter(col("High_Value_Order") != col("prediction"))

display(wrong_preds.select(
    "Order_Date", "Country", "Product_Category", "Customer_Segment",
    "High_Value_Order", "prediction"
))

Order_Date,Country,Product_Category,Customer_Segment,High_Value_Order,prediction
2024-04-23,Australia,Home & Kitchen,Returning,Yes,No
2024-04-25,Pakistan,Sports,Returning,Yes,No
2024-05-11,France,Electronics,Returning,No,Yes
2024-05-13,Pakistan,Electronics,Loyal,No,Yes
2024-05-13,UAE,Fashion,Loyal,Yes,No
2024-05-20,United Kingdom,Fashion,Loyal,Yes,No
2024-05-24,India,Home & Kitchen,New,Yes,No
2024-05-24,United Kingdom,Sports,Premium,Yes,No
2024-05-24,UAE,Home & Kitchen,Loyal,No,Yes
2024-05-26,Canada,Home & Kitchen,Premium,Yes,No


In [0]:
preds_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.ecommerce_predictions")

In [0]:
print(df.count())

5507
